# 02 · EDA del abandono: la foto del asociado

**Diseño:** se observa a cada asociado en el corte **t0 = C** (ahí se miden todas sus variables) y se pregunta
si se retira en (C, C + 3 meses].

- **Población:** asociados activos al corte (en el extracto real, todos tienen `ESTADO` = A).
- **Objetivo:** `EVENTO_RETIRO_3M_SINTETICO`, disponible únicamente en los datos generados. `#RETIROS` es historia acumulada y queda como predictor.
- **Variables:** EDAD y las antigüedades calculadas al corte (`abandono.foto` + `abandono.preparar`).

Si `DATOS_ASOCIADOS` tiene pocas filas, este notebook usa los **mismos 5.000 asociados de desarrollo con que se
elige el campeón** (`campeon.datos_eleccion`, semilla 7, tasa objetivo sintética del 10 %; `make datos-desarrollo`
los deja en `data/processed/`). La etiqueta generada no corresponde a retiros observados del fondo.

> **Datos sintéticos:** las tasas y las diferencias entre los que se quedan y los que se van las fija
> `generador.py`. Sirven para construir y probar el análisis, **no son evidencia sobre el fondo**.

> **Definición vigente (contrato v2.2.0).** Todas las variables están disponibles al corte C = 31-mar-2026; la
> foto es t0 = C y la etiqueta sintética representa un evento posterior en (C, C + 3 meses]. La foto no filtra
> filas. Los hallazgos al final se actualizaron con el generador actual; ninguna cifra sintética es evidencia del fondo.

In [ ]:
import plotly.express as px

from proyecto_ml import abandono as a
from proyecto_ml import campeon as c
from proyecto_ml.config import aviso_origen, get_settings

HORIZONTE = a.HORIZONTE_MESES  # 3 meses: el mismo de make campeon
COLORES = {"se queda": "#2a78d6", "se retira": "#eb6834"}  # paleta de evaluate.py, en orden fijo

df = a.cargar(get_settings().datos_asociados)
if len(df) < 500:
    print(f"{len(df)} filas en DATOS_ASOCIADOS: muy pocas para la foto. Se usan los datos de desarrollo.")
    df, fuente = c.datos_eleccion(origen="sintetico")  # los mismos 5.000 de make campeon
    print(aviso_origen("sintetico"))
print(f"{len(df)} asociados, corte {a.FECHA_CORTE.date()}")

## 1. La foto: t0 = corte y el evento en los 3 meses siguientes

En v2 la foto no deja a nadie por fuera: todos los asociados del extracto están activos al corte, que es donde se miden sus variables. El evento a predecir (aquí, el sintético) cae en (t0, t0 + 3 meses].

In [ ]:
resumen = a.resumen_foto(df)
resumen.round(3)

## 2. La foto elegida: población y objetivo

In [ ]:
vista = a.foto(df, HORIZONTE)
X = a.preparar(vista)  # variables medidas en t0 = corte
evento = a.objetivo_binario(vista)
clase = evento.map({0: "sin evento futuro", 1: "con evento futuro"}).rename("clase")
print(
    f"t0 = {a.fecha_observacion().date()}, evento hasta {a.fin_horizonte(HORIZONTE).date()} · {len(vista)} asociados · "
    f"{evento.sum()} tienen evento futuro sintético ({evento.mean():.1%})"
)
clase.value_counts().to_frame("asociados")

## 3. Comportamiento con el fondo: promedio por clase

Los conteos de solicitudes son la señal principal. **Pregunta para Edwin:** ¿se miden a la fecha de extracción o
tienen fecha? Si son a la extracción, para los retirados incluyen lo que pasó después de t0 (fuga de información).

In [ ]:
promedios = X[a.COMPORTAMIENTO].groupby(clase).mean().T.rename_axis("variable").reset_index()
largo = promedios.melt(id_vars="variable", var_name="clase", value_name="promedio")
fig = px.bar(
    largo,
    y="variable",
    x="promedio",
    color="clase",
    barmode="group",
    orientation="h",
    color_discrete_map=COLORES,
    labels={"promedio": "Promedio por asociado", "variable": ""},
    title=f"Comportamiento promedio por clase (t0 = corte, evento a {HORIZONTE} meses)",
    height=520,
)
fig.update_layout(template="plotly_white", legend_title_text="")
fig

## 4. Perfil: edad, antigüedad y salario por clase

In [ ]:
for columna in ["EDAD", "ANTIGUEDAD_F_MESES", "ANTIGUEDAD_LA_MESES", "SALARIO"]:
    fig = px.box(
        X.assign(clase=clase),
        x="clase",
        y=columna,
        color="clase",
        color_discrete_map=COLORES,
        points=False,
        title=f"{columna} por clase (a t0)",
    )
    fig.update_layout(template="plotly_white", showlegend=False)
    fig.show()

## 5. Categóricas: tasa de retiro por categoría

In [ ]:
for columna in ["TIPOCONTRATO", "SEXO", "ESTADOCIVIL", "ESTRATO"]:
    tasa = (
        vista.assign(**{columna: X[columna], "evento_retiro": evento})
        .groupby(columna)["evento_retiro"]
        .agg(["mean", "size"])
        .reset_index()
    )
    fig = px.bar(
        tasa,
        x=columna,
        y="mean",
        hover_data={"size": True},
        labels={"mean": "Tasa de retiro", "size": "asociados"},
        title=f"Tasa de retiro por {columna}",
        color_discrete_sequence=[COLORES["se retira"]],
    )
    fig.update_yaxes(tickformat=".0%")
    fig.update_layout(template="plotly_white")
    fig.show()

## 6. Hallazgos y límites (actualizado 25-nov-2026)

- **Diseño temporal:** C = 31-mar-2026, variables medidas hasta C y etiqueta sintética en (C, 30-jun-2026].
  El generador actual produce 503 etiquetas positivas entre 5.000 filas (10,1 %); no son retiros observados.
- **Dependencia de la regla:** cancelaciones, retiros parciales, disminución de cuotas y productos se generan
  condicionados a la etiqueta. La separación observada en tablas y gráficos refleja en parte esa construcción.
- **Fondo real:** no hay etiquetas posteriores al corte. `#RETIROS` es historia acumulada de personas activas,
  no el evento futuro; por ello el extracto no permite evaluar el clasificador de abandono.
- **Reproducción:** las métricas vigentes del campeón y de las técnicas están en `docs/metodologia.md` y en
  `reports/metrics/`; son experimentos sintéticos, no validación real.